# SchemaGuard AI

Evidence-backed extraction and validation for electronics warranty claims.

**Pipeline:** SGLang serves Qwen3-1.7B with JSON-schema constrained decoding → a deterministic Python gate accepts a field only if its evidence is quoted verbatim from the claim → CuratorKIT turns the gate-approved outputs into a training set → Streamlit demo.

**Cell order:** 1 Environment · 2 SGLang server · 3 Core module · 4 Benchmark · 5 Curation · 6 Demo

## 1. Environment

In [ ]:
import sys, torch

print("Python:", sys.version.split()[0], "| PyTorch:", torch.__version__, "| CUDA:", torch.version.cuda)
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f"GPU {i}: {p.name} ({p.total_memory / 1024**3:.1f} GiB)")

!nvcc --version | tail -n 1   # needed for FlashInfer JIT compilation

SGLang gets its own Python 3.12 venv so its pinned PyTorch cannot clash with the notebook environment (takes about 5 minutes the first time).

In [ ]:
!test -x /kaggle/working/sglang-venv/bin/python || uv venv --python 3.12 /kaggle/working/sglang-venv
!UV_LINK_MODE=copy uv pip install --python /kaggle/working/sglang-venv/bin/python --prerelease=allow sglang==0.5.22
!/kaggle/working/sglang-venv/bin/python -c "import torch, sglang; print('PyTorch', torch.__version__, '| CUDA', torch.cuda.is_available(), '| SGLang import OK')"

## 2. SGLang server

Fixes needed on Kaggle T4s:
- `libcuda.so` symlink and `LD_LIBRARY_PATH`, so FlashInfer's JIT-compiled kernels can link;
- `--disable-cuda-graph` and a reduced memory fraction / context length.

The first start compiles kernels for `sm_75` and can take several minutes. Re-running this cell is safe: it does nothing if the server is already healthy.

In [ ]:
import os, subprocess, time
from pathlib import Path
import requests

WORK = Path("/kaggle/working")
SGLANG_PY = WORK / "sglang-venv/bin/python"
SGLANG_LOG = WORK / "sglang-server.log"
HEALTH_URL = "http://127.0.0.1:30000/health"
MODEL = "Qwen/Qwen3-1.7B"


def sglang_healthy():
    try:
        return requests.get(HEALTH_URL, timeout=3).status_code == 200
    except requests.RequestException:
        return False


def start_sglang(model=MODEL, gpu="0", wait_seconds=900):
    if sglang_healthy():
        print("SGLang already running.")
        return

    subprocess.run("pkill -f '[s]glang.launch_server' || true", shell=True)

    # Linker stub that FlashInfer's JIT build expects.
    stub, driver = Path("/usr/local/cuda/lib64/libcuda.so"), Path("/usr/local/nvidia/lib64/libcuda.so.1")
    if driver.exists() and not stub.exists():
        stub.symlink_to(driver)

    env = os.environ.copy()
    env["CUDA_VISIBLE_DEVICES"] = gpu
    env["LD_LIBRARY_PATH"] = "/usr/local/nvidia/lib64:/usr/local/cuda/lib64:" + env.get("LD_LIBRARY_PATH", "")

    subprocess.Popen(
        [str(SGLANG_PY), "-m", "sglang.launch_server",
         "--model-path", model,
         "--host", "127.0.0.1", "--port", "30000",
         "--mem-fraction-static", "0.65",
         "--context-length", "2048",
         "--disable-cuda-graph"],
        env=env, stdout=open(SGLANG_LOG, "w"), stderr=subprocess.STDOUT,
        start_new_session=True,
    )

    deadline = time.time() + wait_seconds
    while time.time() < deadline:
        if sglang_healthy():
            print("SGLang healthy.")
            return
        time.sleep(5)
    print("SGLang did not become healthy. Log tail:\n", SGLANG_LOG.read_text(errors="replace")[-4000:])


start_sglang()

## 3. Core module

Extraction and validation live in one importable file, so the notebook and the Streamlit demo run **the same logic**.

- The model returns, per field, a `value` and a verbatim `evidence` quote (flat schema, constrained by SGLang).
- Validation is plain Python, not the model: evidence and value must occur in the claim; ambiguous numeric dates (e.g. `03/04/2026`) or several different dates withhold `purchase_date`.

In [ ]:
%%writefile /kaggle/working/schemaguard.py
"""SchemaGuard core: SGLang-backed extraction plus deterministic validation."""
import json
import re

import requests

URL = "http://127.0.0.1:30000/v1/chat/completions"
MODEL = "Qwen/Qwen3-1.7B"

FIELDS = ["purchase_date", "product", "issue", "serial_number"]
FLAT_KEYS = [f"{field}_{part}" for field in FIELDS for part in ("value", "evidence")]

# Flat schema: eight strings, no model-generated statuses or conflicts.
SCHEMA = {
    "type": "object",
    "properties": {key: {"type": "string"} for key in FLAT_KEYS},
    "required": FLAT_KEYS,
    "additionalProperties": False,
}

SYSTEM_PROMPT = (
    "Extract purchase_date, product, issue, serial_number. "
    "Return exactly the flat JSON fields in the schema. "
    "Copy short supporting evidence verbatim from the claim. "
    "Use empty strings for fields not explicitly stated. "
    "Do not invent information. Preserve date wording. "
    "The issue is a reported defect, not a warranty request. "
    "Do not generate statuses, confidence scores, or conflicts."
)


# --------------------------------------------------------------- extraction
def extract_claim(claim, model=MODEL, system_prompt=SYSTEM_PROMPT, url=URL):
    """Call SGLang with strict JSON-schema decoding; return {field: {value, evidence}}."""
    payload = {
        "model": model,
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": f"Claim:\n{claim}"},
        ],
        "temperature": 0,
        "max_tokens": 300,
        "chat_template_kwargs": {"enable_thinking": False},
        "response_format": {
            "type": "json_schema",
            "json_schema": {"name": "claim_extraction", "strict": True, "schema": SCHEMA},
        },
    }
    response = requests.post(url, json=payload, timeout=120)
    response.raise_for_status()
    body = response.json()
    choice = body["choices"][0]

    if choice.get("finish_reason") == "length":
        raise RuntimeError(f"Truncated generation; usage={body.get('usage')}")

    flat = json.loads(choice["message"]["content"])
    if set(flat) != set(FLAT_KEYS):
        raise ValueError(f"Unexpected output keys: {list(flat)}")
    if not all(isinstance(v, str) for v in flat.values()):
        raise ValueError("All output fields must be strings")

    return {
        field: {
            "value": flat[f"{field}_value"].strip(),
            "evidence": flat[f"{field}_evidence"].strip(),
        }
        for field in FIELDS
    }


# --------------------------------------------------------------- validation
def norm(text):
    return re.sub(r"[^a-z0-9]+", " ", str(text).casefold()).strip()


def occurs(source, span):
    """True if `span` appears in `source` as a contiguous run of whole tokens."""
    span = norm(span)
    return bool(span) and f" {span} " in f" {norm(source)} "


MONTHS = (
    "January|February|March|April|May|June|July|August|September|"
    "October|November|December|Jan|Feb|Mar|Apr|Jun|Jul|Aug|Sep|Sept|"
    "Oct|Nov|Dec"
)
DATE_RE = re.compile(
    rf"\b(?:\d{{4}}-\d{{2}}-\d{{2}}"
    rf"|\d{{1,2}}[/-]\d{{1,2}}[/-]\d{{2,4}}"
    rf"|\d{{1,2}}\s+(?:{MONTHS})\s+\d{{4}}"
    rf"|(?:{MONTHS})\s+\d{{1,2}},?\s+\d{{4}}"
    rf"|(?:{MONTHS})\s+\d{{4}})\b",
    re.IGNORECASE,
)
NUMERIC_DATE_RE = re.compile(r"\b(\d{1,2})[/-](\d{1,2})[/-](\d{2,4})\b")

# Small domain vocabulary guard for the `issue` field; not a general defect classifier.
DEFECT_TERMS = [
    "charging", "flicker", "flickers", "flickering",
    "battery drains", "drains quickly", "overheats", "overheating",
    "broken", "cracked", "damaged", "stopped working",
    "not turning on", "wont start",
]


def classify_dates(claim):
    unique = list(dict.fromkeys(norm(m.group(0)) for m in DATE_RE.finditer(claim)))

    for m in NUMERIC_DATE_RE.finditer(claim):
        a, b = int(m.group(1)), int(m.group(2))
        if 1 <= a <= 12 and 1 <= b <= 12 and a != b:
            return "AMBIGUOUS_DATE_REVIEW"
    if len(unique) > 1:
        return "MULTIPLE_DATES_REVIEW"
    return "NO_DATE_REVIEW_TRIGGERED"


def validate_claim(claim, extracted):
    """Accept a field only if its evidence and value are literally present in the claim."""
    date_check = classify_dates(claim)
    claim_norm = norm(claim)
    final_values, statuses, details = {}, {}, {}

    for field in FIELDS:
        value, evidence = extracted[field]["value"].strip(), extracted[field]["evidence"].strip()

        if not value and not evidence:
            status = "MISSING"
        elif not value or not evidence:
            status = "REVIEW"
        elif not occurs(claim, evidence):
            status = "INVALID_EVIDENCE"
        elif not occurs(claim, value):
            status = "VALUE_NOT_LITERAL_REVIEW"
        elif field == "issue" and not any(term in claim_norm for term in DEFECT_TERMS):
            status = "NO_ISSUE_STATED"
        else:
            status = "EVIDENCE_PRESENT"

        accepted = value if status == "EVIDENCE_PRESENT" else ""

        # Date ambiguity or several dates overrides any candidate purchase date.
        if field == "purchase_date" and date_check != "NO_DATE_REVIEW_TRIGGERED":
            status, accepted = date_check, ""

        final_values[field] = accepted
        statuses[field] = status
        details[field] = {
            "value": value,
            "evidence": evidence,
            "status": status,
            "evidence_in_source": occurs(claim, evidence),
            "value_in_source": occurs(claim, value),
        }

    review = any(s != "EVIDENCE_PRESENT" for s in statuses.values())
    return {
        "final_values": final_values,
        "field_statuses": statuses,
        "field_details": details,
        "date_check": date_check,
        "overall_status": "REVIEW_REQUIRED" if review else "BASIC_CHECKS_PASSED",
    }

In [ ]:
import sys, json
sys.path.insert(0, "/kaggle/working")
import importlib, schemaguard
importlib.reload(schemaguard)
from schemaguard import FIELDS, norm, occurs, extract_claim, validate_claim

claim = "Bought my laptop on 12 February 2026. It stopped charging after 8 months. Serial number: SN123456."
extracted = extract_claim(claim)
result = validate_claim(claim, extracted)

print(json.dumps(extracted, indent=2))
print(result["overall_status"], "|", result["date_check"])
print(json.dumps(result["final_values"], indent=2))

## 4. Benchmark

50 generated claims across six categories. Gold answers are blank wherever the system should **abstain** (missing fields, ambiguous or conflicting dates, ticket dates that are not purchase dates).

In [ ]:
DEVICES = [
    ("laptop", "overheats"), ("tablet", "battery drains quickly"), ("monitor", "screen flickers"),
    ("printer", "stopped working"), ("phone", "stopped charging"),
]
NORMAL_DATES = ["12 February 2026", "March 2026", "May 15, 2025", "22 October 2024"]
AMBIGUOUS_DATES = ["03/04/2026", "04/05/2026", "06/07/2026", "08/09/2026", "11/12/2026"]
CONFLICTING_DATES = [
    ("12 February 2026", "15 February 2026"), ("3 March 2026", "8 March 2026"),
    ("May 15, 2025", "May 20, 2025"), ("7 July 2025", "9 July 2025"),
    ("October 2, 2024", "October 5, 2024"),
]
TICKET_DATES = ["12 February 2026", "March 2026", "May 15, 2025", "22 October 2024", "January 9, 2025"]


def gold(product, issue, date="", serial=""):
    """Accepted answers per field; several phrasings are allowed for `issue`."""
    return {
        "purchase_date": [date],
        "product": [product],
        "issue": [issue, f"My {product} {issue}"],
        "serial_number": [serial],
    }


def build_benchmark():
    cases = []

    def add(category, claim, answers):
        cases.append({"id": f"B{len(cases) + 1:03}", "category": category, "claim": claim, "gold": answers})

    for product, issue in DEVICES:                       # 20 ordinary claims
        for date in NORMAL_DATES:
            serial = f"SN{10000 + len(cases) + 1}"
            add("normal", f"My {product} {issue}. Purchased on {date}. Serial number: {serial}.",
                gold(product, issue, date, serial))

    for j in range(10):                                  # 10 with no date and no serial
        product, issue = DEVICES[j % len(DEVICES)]
        add("missing_fields", f"My {product} {issue}. Please repair it under warranty.", gold(product, issue))

    for j, (product, issue) in enumerate(DEVICES):       # 5 with a date but no serial
        date = NORMAL_DATES[j % len(NORMAL_DATES)]
        add("missing_serial", f"My {product} {issue}. Purchased on {date}.", gold(product, issue, date))

    for j, date in enumerate(AMBIGUOUS_DATES):           # 5 ambiguous numeric dates
        product, issue = DEVICES[j % len(DEVICES)]
        serial = f"AM{20000 + j}"
        add("ambiguous_date",
            f"I purchased my {product} on {date}. My {product} {issue}. Serial number: {serial}.",
            gold(product, issue, "", serial))

    for j, (d1, d2) in enumerate(CONFLICTING_DATES):     # 5 contradictory purchase/invoice dates
        product, issue = DEVICES[j % len(DEVICES)]
        serial = f"CF{30000 + j}"
        add("conflicting_dates",
            f"My {product} was purchased on {d1}. The invoice lists {d2}. "
            f"My {product} {issue}. Serial number: {serial}.",
            gold(product, issue, "", serial))

    for j, date in enumerate(TICKET_DATES):              # 5 dates that are NOT the purchase date
        product, issue = DEVICES[j % len(DEVICES)]
        serial = f"MT{40000 + j}"
        add("unrelated_date",
            f"My {product} {issue}. Support ticket created on {date}. "
            f"Purchase date was not provided. Serial number: {serial}.",
            gold(product, issue, "", serial))

    assert len(cases) == 50, len(cases)
    return cases


benchmark = build_benchmark()
print({c: sum(x["category"] == c for x in benchmark) for c in sorted({x["category"] for x in benchmark})})

In [ ]:
RESULTS_PATH = WORK / "schemaguard_benchmark_50.json"


def run_benchmark(cases):
    rows = []
    for case in cases:
        row = {"id": case["id"], "category": case["category"], "claim": case["claim"],
               "extracted": None, "decision": None, "error": None}
        try:
            row["extracted"] = extract_claim(case["claim"])
            row["decision"] = validate_claim(case["claim"], row["extracted"])
        except Exception as exc:
            row["error"] = str(exc)
        rows.append(row)
    return rows


results = run_benchmark(benchmark)
RESULTS_PATH.write_text(json.dumps(results, indent=2))
print(f"Saved {len(results)} rows to {RESULTS_PATH} | failures: {sum(r['error'] is not None for r in results)}")

In [ ]:
from collections import Counter, defaultdict

DATE_FLAG = {"ambiguous_date": "AMBIGUOUS_DATE_REVIEW", "conflicting_dates": "MULTIPLE_DATES_REVIEW"}


def evaluate(rows, cases):
    answers = {c["id"]: c for c in cases}
    by_field, by_category = defaultdict(lambda: [0, 0]), defaultdict(lambda: [0, 0])
    m = Counter()

    for row in rows:
        if row["decision"] is None:
            m["failed"] += 1
            continue
        case = answers[row["id"]]

        for field in FIELDS:
            accepted = {norm(v) for v in case["gold"][field]}
            actual = norm(row["decision"]["final_values"][field])
            hit = actual in accepted
            for table, key in ((by_field, field), (by_category, case["category"])):
                table[key][0] += hit
                table[key][1] += 1

            if accepted == {""}:                         # field should be left blank
                m["expected_blank"] += 1
                m["false_fills" if actual else "withheld"] += 1

            evidence = row["extracted"][field]["evidence"]
            if evidence:
                m["evidence_total"] += 1
                m["evidence_in_source"] += evidence.casefold() in case["claim"].casefold()

        if case["category"] in DATE_FLAG:
            m["date_cases"] += 1
            m["date_flagged"] += row["decision"]["date_check"] == DATE_FLAG[case["category"]]

    return by_field, by_category, m


def pct(a, b):
    return f"{a}/{b} ({a / b:.1%})" if b else "n/a"


by_field, by_category, m = evaluate(results, benchmark)
print("SCHEMAGUARD BENCHMARK")
print("Claims evaluated:", len(results) - m["failed"], "/", len(results))
print("Accepted-value accuracy:", pct(sum(c for c, _ in by_field.values()), sum(t for _, t in by_field.values())))
print("\nBy field:")
for k, (c, t) in by_field.items():
    print(f"  {k:14s} {pct(c, t)}")
print("\nBy category:")
for k, (c, t) in sorted(by_category.items()):
    print(f"  {k:18s} {pct(c, t)}")
print("\nAbstention:")
print("  Blanks correctly withheld:", pct(m["withheld"], m["expected_blank"]))
print("  False fills:", m["false_fills"])
print("Evidence found verbatim in claim:", pct(m["evidence_in_source"], m["evidence_total"]))
print("Date cases correctly flagged:", pct(m["date_flagged"], m["date_cases"]))

Where do the errors come from? Counts of wrong accepted values by field and validator status.

In [ ]:
def error_breakdown(rows, cases):
    answers = {c["id"]: c for c in cases}
    errors = Counter()
    for row in rows:
        if row["decision"] is None:
            continue
        case = answers[row["id"]]
        for field in FIELDS:
            accepted = {norm(v) for v in case["gold"][field]}
            if norm(row["decision"]["final_values"][field]) not in accepted:
                errors[(field, row["decision"]["field_statuses"][field])] += 1
    return errors


for (field, status), n in error_breakdown(results, benchmark).most_common():
    print(f"{n:3d}  {field:14s} {status}")

## 5. Curation with CuratorKIT

Only gate-approved **normal** claims become training candidates; everything else is held out for review. These labels are rule-checked model outputs, not human-verified ground truth. CuratorKIT then applies its schema gate, exact dedup and cleaning, and exports Alpaca/ShareGPT files.

In [ ]:
%pip install -q curatorkit nest_asyncio

In [ ]:
from curatorkit import Curator, CuratorConfig

CURATION_DIR = WORK / "schemaguard_curation"
CURATION_DIR.mkdir(exist_ok=True)

INSTRUCTION = (
    "Extract purchase_date, product, issue, and serial_number from the claim. "
    "Return JSON with each field containing value and evidence. "
    "Copy evidence from the claim verbatim. "
    "Use empty strings for information not explicitly stated."
)


def curation_reasons(row):
    """Empty list means the row is a clean training candidate."""
    if row["error"] or row["decision"] is None:
        return ["INFERENCE_OR_VALIDATION_ERROR"]

    reasons = []
    if row["category"] != "normal":
        reasons.append("NON_NORMAL_CASE_HELD_OUT")
    if row["decision"]["overall_status"] != "BASIC_CHECKS_PASSED":
        reasons.append("BASIC_CHECKS_NOT_PASSED")

    for field in FIELDS:
        value, evidence = row["extracted"][field]["value"], row["extracted"][field]["evidence"]
        if row["decision"]["field_statuses"][field] != "EVIDENCE_PRESENT":
            reasons.append(f"{field}:STATUS_NOT_ACCEPTED")
        if not value:
            reasons.append(f"{field}:MISSING_VALUE")
        elif not occurs(row["claim"], value):
            reasons.append(f"{field}:VALUE_NOT_IN_SOURCE")
        if not occurs(row["claim"], evidence):
            reasons.append(f"{field}:INVALID_EVIDENCE")
        if norm(row["decision"]["final_values"][field]) != norm(value):
            reasons.append(f"{field}:FINAL_VALUE_MISMATCH")
    return sorted(set(reasons))


def write_jsonl(path, rows):
    path.write_text("".join(json.dumps(r, ensure_ascii=False) + "\n" for r in rows), encoding="utf-8")


rows = json.loads(RESULTS_PATH.read_text(encoding="utf-8"))
train, held_out = [], []
for row in rows:
    reasons = curation_reasons(row)
    if reasons:
        held_out.append({"id": row["id"], "category": row["category"], "claim": row["claim"], "reasons": reasons})
    else:
        train.append({
            "instruction": INSTRUCTION,
            "input": row["claim"],
            "output": json.dumps({f: row["extracted"][f] for f in FIELDS}, ensure_ascii=False),
        })

seed_path = CURATION_DIR / "schemaguard_seed.jsonl"
write_jsonl(seed_path, train)
write_jsonl(CURATION_DIR / "held_out_review.jsonl", held_out)
print(f"Records: {len(rows)} | training candidates: {len(train)} | held out: {len(held_out)}")

assert train, "No training candidates; inspect held_out_review.jsonl"

result = Curator(CuratorConfig(
    dataset=str(seed_path),
    format="alpaca",
    schema_gate=True,
    schema_required_fields=["instruction", "input", "output"],
    min_tokens=1,
    max_tokens=2048,
    dedup="exact",
    clean=True,
    export_formats=["alpaca", "sharegpt"],
    output_dir=str(CURATION_DIR / "official_run"),
    name="SchemaGuard_CuratorKIT",
)).run()
result.print_summary()
print(sorted(p.name for p in (CURATION_DIR / "official_run").glob("*")))

## 6. Demo

A Streamlit app (run inside the SGLang venv) calls the same `extract_claim` / `validate_claim`, and a Cloudflare quick tunnel exposes it publicly. Keep this Kaggle session running while demoing.

In [ ]:
!UV_LINK_MODE=copy uv pip install -q --python /kaggle/working/sglang-venv/bin/python streamlit

In [ ]:
%%writefile /kaggle/working/app.py
import json
import sys
from pathlib import Path

import requests
import streamlit as st

sys.path.insert(0, str(Path(__file__).resolve().parent))
from schemaguard import MODEL, extract_claim, validate_claim

CURATED_FILE = Path("/kaggle/working/schemaguard_curation/official_run/sft_alpaca.jsonl")
SAMPLE = (
    "My laptop overheats during video calls. "
    "Purchased on 12 February 2026. Serial number: SN10001."
)

st.set_page_config(page_title="SchemaGuard AI", page_icon="🛡️", layout="wide")
st.title("🛡️ SchemaGuard AI")
st.caption("Evidence-backed extraction and validation for electronics warranty claims")

with st.sidebar:
    st.header("System")
    st.write("Inference engine: SGLang")
    st.write(f"Model: `{MODEL}`")
    st.write("Curation: Lexsi Labs CuratorKIT")
    if CURATED_FILE.exists():
        with CURATED_FILE.open(encoding="utf-8") as f:
            st.metric("Curated candidate examples", sum(1 for line in f if line.strip()))
        st.caption("CuratorKIT output; labels are not independently verified.")

claim = st.text_area(
    "Enter a warranty claim", value=SAMPLE, height=150,
    help="Try a claim with missing information or conflicting dates to trigger review.",
)

if st.button("Extract and validate", type="primary"):
    if not claim.strip():
        st.warning("Enter a claim first.")
    else:
        try:
            with st.spinner("Running extraction and validation..."):
                extracted = extract_claim(claim)
                result = validate_claim(claim, extracted)

            if result["overall_status"] == "BASIC_CHECKS_PASSED":
                st.success("Basic validation checks passed.")
            else:
                st.warning("Human review required.")

            left, right = st.columns(2)
            with left:
                st.subheader("Accepted values")
                st.json(result["final_values"])
            with right:
                st.subheader("Validation status")
                st.json({
                    "overall_status": result["overall_status"],
                    "date_check": result["date_check"],
                    "field_statuses": result["field_statuses"],
                })

            st.subheader("Evidence and field checks")
            st.dataframe([
                {
                    "Field": field,
                    "Extracted value": d["value"],
                    "Evidence": d["evidence"],
                    "Status": d["status"],
                    "Evidence in claim": d["evidence_in_source"],
                    "Value in claim": d["value_in_source"],
                }
                for field, d in result["field_details"].items()
            ], hide_index=True)

            with st.expander("Raw extraction"):
                st.json(extracted)

            st.download_button(
                "Download result as JSON",
                data=json.dumps({"claim": claim, "extracted": extracted, "decision": result},
                                indent=2, ensure_ascii=False),
                file_name="schemaguard_result.json",
                mime="application/json",
            )
        except requests.RequestException as exc:
            st.error("Could not reach SGLang on port 30000. Check that the server is running.")
            st.code(str(exc))
        except Exception as exc:
            st.error(f"Extraction or validation failed: {exc}")

In [ ]:
import re

CLOUDFLARED = WORK / "cloudflared"

subprocess.run("pkill -f '[s]treamlit run' || true; pkill -f '[c]loudflared tunnel' || true", shell=True)

subprocess.Popen(
    [str(SGLANG_PY), "-m", "streamlit", "run", str(WORK / "app.py"),
     "--server.address", "0.0.0.0", "--server.port", "8501", "--server.headless", "true"],
    cwd=WORK, stdout=open(WORK / "streamlit.log", "w"), stderr=subprocess.STDOUT, start_new_session=True,
)

for _ in range(30):
    try:
        if requests.get("http://127.0.0.1:8501/_stcore/health", timeout=2).ok:
            break
    except requests.RequestException:
        time.sleep(2)
else:
    raise RuntimeError((WORK / "streamlit.log").read_text()[-3000:])

if not CLOUDFLARED.exists():
    subprocess.run(["wget", "-q", "-O", str(CLOUDFLARED),
                    "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"],
                   check=True)
    CLOUDFLARED.chmod(0o755)

tunnel_log = WORK / "cloudflared.log"
subprocess.Popen([str(CLOUDFLARED), "tunnel", "--url", "http://127.0.0.1:8501", "--no-autoupdate"],
                 stdout=open(tunnel_log, "w"), stderr=subprocess.STDOUT, start_new_session=True)

public_url = None
for _ in range(30):
    time.sleep(2)
    found = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", tunnel_log.read_text(errors="replace"))
    if found:
        public_url = found.group(0)
        break

print("SGLang:", "healthy" if sglang_healthy() else "DOWN")
print("DEMO URL:", public_url or tunnel_log.read_text(errors="replace")[-3000:])

## Not done yet

- Fine-tune (AlignTune) on a larger, ground-truth training set that includes abstention cases.
- Compare base vs base+gate vs fine-tuned vs fine-tuned+gate on held-out claims.
- Run AuditKIT evaluations against the SGLang endpoint.